# Notebook 1 — Data Cleaning

This notebook loads and cleans all raw data sources so they are consistent and ready to use in later notebooks.

**Outputs:**
- `output/results_clean.csv` — cleaned match history with normalised team names
- `output/historical_elo_clean.csv` — ELO ratings per team per date snapshot (1872–2025)
- `output/wc2026_elo_clean.csv` — ELO ratings for all 48 2026 WC teams (May 2026 snapshot)

---
## 1. Imports & Paths

In [110]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR   = Path('international_matches')
WC_DIR     = Path('world_cup_matches')
ELO_DIR    = Path('elos')
OUTPUT_DIR = Path('output')
OUTPUT_DIR.mkdir(exist_ok=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

---
## 2. Load Raw Data

| File | Description |
|---|---|
| `results.csv` | Every international match from 1872 to 2026 (~49 k rows). Primary source for form features. |
| `former_names.csv` | Maps historical team names to their modern equivalents. |
| `matches_1930_2022.csv` | Detailed World Cup match data for model training and validation. |
| `historical_elo_ratings.csv` | ELO rating per team per date snapshot, 1872–Dec 2025. |
| `wc2026_elo_ratings.csv` | ELO snapshot for all 48 confirmed 2026 WC teams (May 27 2026). |

In [111]:
results      = pd.read_csv(DATA_DIR / 'results.csv', parse_dates=['date'])
former_names = pd.read_csv(DATA_DIR / 'former_names.csv')
wc_matches   = pd.read_csv(WC_DIR   / 'matches_1930_2022.csv')
hist_elo     = pd.read_csv(ELO_DIR  / 'historical_elo_ratings.csv')
wc2026_elo   = pd.read_csv(ELO_DIR  / 'wc2026_elo_ratings.csv')

print('results:      ', results.shape)
print('former_names: ', former_names.shape)
print('wc_matches:   ', wc_matches.shape)
print('hist_elo:     ', hist_elo.shape)
print('wc2026_elo:   ', wc2026_elo.shape)

results:       (49450, 9)
former_names:  (36, 4)
wc_matches:    (964, 44)
hist_elo:      (6678, 4)
wc2026_elo:    (48, 6)


---
## 3. Clean `results.csv`

### 3a. Drop rows with missing scores

72 rows have no score recorded — either future fixtures or data gaps. We drop them since we can only compute form from completed matches.

In [112]:
print('Rows with missing scores:', results[['home_score', 'away_score']].isnull().any(axis=1).sum())
results = results.dropna(subset=['home_score', 'away_score']).copy()

results['home_score'] = results['home_score'].astype(int)
results['away_score'] = results['away_score'].astype(int)

print('Remaining rows:', len(results))

Rows with missing scores: 72
Remaining rows: 49378


### 3b. Normalise team names using `former_names.csv`

Some teams changed their name over time (Soviet Union → Russia, West Germany → Germany, etc.).
We map every historical name to its current successor so form calculations are continuous.

In [113]:
# Columns: current, former, start_date, end_date
former_names.head()

,current,former,start_date,end_date
0,Benin,Dahomey,1959-11-08,1975-11-30
1,Burkina Faso,Upper Volta,1960-04-14,1984-08-04
2,Curaçao,Netherlands Antilles,1957-03-03,2010-10-10
3,Czechoslovakia,Bohemia,1903-04-05,1919-01-01
4,Czechoslovakia,Bohemia and Moravia,1939-01-01,1945-05-01


In [114]:
name_map = dict(zip(former_names['former'], former_names['current']))
print(f'Name mappings loaded: {len(name_map)}')

results['home_team'] = results['home_team'].replace(name_map)
results['away_team'] = results['away_team'].replace(name_map)

all_teams = pd.unique(results[['home_team', 'away_team']].values.ravel())
print(f'Unique teams after normalisation: {len(all_teams)}')

Name mappings loaded: 36
Unique teams after normalisation: 336


### 3c. Sort chronologically

Form is computed by looking at the last N matches per team, so the data must be in date order.

In [115]:
results = results.sort_values('date').reset_index(drop=True)
print('Date range:', results['date'].min().date(), '→', results['date'].max().date())

Date range: 1872-11-30 → 2026-06-08


### 3d. Quick data quality check

In [116]:
print('Missing values:')
print(results.isnull().sum())
print()
results.sample(5, random_state=42)

Missing values:
date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64



,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
18148,1992-02-19,Spain,Russia,1,1,Friendly,Valencia,Spain,False
10072,1975-04-02,Italy,United States,10,0,Friendly,Rome,Italy,False
43809,2021-03-27,Lesotho,Sierra Leone,0,0,African Cup of Nations qualification,Maseru,Lesotho,False
38841,2015-06-16,Poland,Greece,0,0,Friendly,Gdańsk,Poland,False
42550,2019-06-09,Mexico,Ecuador,3,2,Friendly,Arlington,United States,True


---
## 4. Clean `historical_elo_ratings.csv`

This file contains a time-series ELO rating for every team from 1872 through December 2025.
Each row is one team's rating on a given date.

Two things to fix:
1. **Date format** — the raw file uses `MM/DD/YYYY` for recent rows and `YYYY-MM-DD` for older ones. We normalise everything to `YYYY-MM-DD`.
2. **Team name alignment** — names here must match `results.csv` so we can join them later.

In [117]:
print(hist_elo.dtypes)
print()
hist_elo.head()

date          str
team          str
rating    float64
change      int64
dtype: object



,date,team,rating,change
0,1872-11-30,England,2003.00,3
1,1872-11-30,Scotland,1997.00,-3
2,1873-03-08,England,2014.00,11
3,1873-03-08,Scotland,1986.00,-11
4,1874-03-07,England,2006.00,-8


In [118]:
# Check the mixed date formats
print('Sample raw date values:')
print(hist_elo['date'].head(5).tolist())
print(hist_elo['date'].tail(5).tolist())

Sample raw date values:
['1872-11-30', '1872-11-30', '1873-03-08', '1873-03-08', '1874-03-07']
['12/13/2025', '12/13/2025', '12/13/2025', '12/13/2025', '12/13/2025']


In [119]:
# The file mixes two date formats: YYYY-MM-DD (older rows) and M/D/YYYY (recent rows)
# format='mixed' tells pandas to infer each value individually
hist_elo['date'] = pd.to_datetime(hist_elo['date'], format='mixed')

print('Date range:', hist_elo['date'].min().date(), '→', hist_elo['date'].max().date())
print('Unique teams:', hist_elo['team'].nunique())
print('Missing values:')
print(hist_elo.isnull().sum())

Date range: 1872-11-30 → 2025-12-13
Unique teams: 270
Missing values:
date       0
team       0
rating    31
change     0
dtype: int64


In [120]:
# Strip non-breaking spaces (\xa0) from team names — encoding issue in the source CSV
hist_elo['team'] = hist_elo['team'].str.replace('\xa0', ' ', regex=False)

# Drop any rows where rating is 0 — placeholder/missing entries
zero_ratings = (hist_elo['rating'] == 0).sum()
print(f'Rows with rating == 0: {zero_ratings}')
hist_elo = hist_elo[hist_elo['rating'] > 0].copy()
print(f'Remaining rows: {len(hist_elo):,}')

Rows with rating == 0: 1
Remaining rows: 6,646


### 4a. Check name alignment with `results.csv`

The form function joins ELO to matches by team name, so any mismatch means a team gets no ELO assigned.

In [121]:
results_teams  = set(pd.unique(results[['home_team', 'away_team']].values.ravel()))
hist_elo_teams = set(hist_elo['team'].unique())

in_elo_not_results = sorted(hist_elo_teams - results_teams)
print(f'In historical ELO but NOT in results.csv ({len(in_elo_not_results)}) — potential mismatches:')
print(in_elo_not_results)

In historical ELO but NOT in results.csv (39) — potential mismatches:
['British Guiana', 'Burma', 'Ceylon', 'Christmas Island', 'Cocos Islands', 'Congo-Brazzaville', 'Czechia', 'Dahomey', 'Democratic Republic of Congo', 'East Germany', 'East Timor', 'Eastern Samoa', 'Federated States of Micronesia', 'Ireland', 'Khmer Republic', 'Macao', 'Macedonia', 'Malaya', 'Netherlands Antilles', 'North Yemen', 'Northern Rhodesia', 'Reunion', 'Saba', 'Saint Barthelemy', 'Sao Tome and Principe', 'Serbia and Montenegro', 'Sint Eustatius', 'South Vietnam', 'Southern Rhodesia', 'Soviet Union', 'Surinam', 'Swaziland', 'Tanganyika', 'US Virgin Islands', 'United Arab Republic', 'Upper Volta', 'Vatican', 'Wallis and Futuna', 'West Germany']


In [122]:
# Fill in mismatches found above: {historical ELO name: results.csv name}
HIST_ELO_NAME_FIX = {
    'Czechia': 'Czech Republic',
    'Democratic Republic of Congo': 'DR Congo',
    'Ireland': 'Republic of Ireland',
    'Macedonia': 'North Macedonia',
    'Serbia and Montenegro': 'Serbia',
    'FR Yugoslavia': 'Serbia',
    'Yugoslavia': 'Serbia',
    'Soviet Union': 'Russia',
    'US Virgin Islands': "United States Virgin Islands",
    'West Germany': 'Germany'
}

if HIST_ELO_NAME_FIX:
    hist_elo['team'] = hist_elo['team'].replace(HIST_ELO_NAME_FIX)

# Verify — print before/after for any key in the fix dict
for old, new in HIST_ELO_NAME_FIX.items():
    still_old = (hist_elo['team'] == old).sum()
    now_new   = (hist_elo['team'] == new).sum()
    print(f"  '{old}' remaining: {still_old} | '{new}' rows: {now_new}")

if not HIST_ELO_NAME_FIX:
    print('No fixes applied.')

  'Czechia' remaining: 0 | 'Czech Republic' rows: 37
  'Democratic Republic of Congo' remaining: 0 | 'DR Congo' rows: 19
  'Ireland' remaining: 0 | 'Republic of Ireland' rows: 60
  'Macedonia' remaining: 0 | 'North Macedonia' rows: 36
  'Serbia and Montenegro' remaining: 0 | 'Serbia' rows: 46
  'FR Yugoslavia' remaining: 0 | 'Serbia' rows: 46
  'Yugoslavia' remaining: 0 | 'Serbia' rows: 46
  'Soviet Union' remaining: 0 | 'Russia' rows: 45
  'US Virgin Islands' remaining: 0 | 'United States Virgin Islands' rows: 11
  'West Germany' remaining: 0 | 'Germany' rows: 49


---
## 5. Clean `wc2026_elo_ratings.csv`

This file has one row per confirmed 2026 WC team with their ELO as of May 27 2026.
We only need `country` (team name) and `rating`. We also check name alignment.

### 5a. Check name alignment with `results.csv`

In [123]:
wc2026_elo = wc2026_elo.rename(columns={'country': 'team', 'rating': 'elo', 'rank': 'elo_rank'})
wc2026_elo = wc2026_elo.sort_values('elo', ascending=False).reset_index(drop=True)

print(f'Teams: {len(wc2026_elo)}')
wc2026_elo

Teams: 48


,year,snapshot_date,team,elo_rank,country_code,elo
0,2026,2026-05-27,Spain,1,ES,2165
1,2026,2026-05-27,Argentina,2,AR,2113
2,2026,2026-05-27,France,3,FR,2081
3,2026,2026-05-27,England,4,EN,2020
4,2026,2026-05-27,Brazil,5,BR,1984
5,2026,2026-05-27,Portugal,5,PT,1984
6,2026,2026-05-27,Colombia,7,CO,1975
7,2026,2026-05-27,Netherlands,8,NL,1961
8,2026,2026-05-27,Ecuador,9,EC,1933
9,2026,2026-05-27,Croatia,10,HR,1930


In [124]:
# Fill in mismatches: {wc2026 name: results.csv name}
WC2026_NAME_FIX = {
    'Czechia': 'Czech Republic',
    'Democratic Republic of Congo': 'DR Congo'
}

if WC2026_NAME_FIX:
    wc2026_elo['team'] = wc2026_elo['team'].replace(WC2026_NAME_FIX)

# Verify — print before/after for any key in the fix dict
for old, new in WC2026_NAME_FIX.items():
    still_old = (wc2026_elo['team'] == old).sum()
    now_new   = (wc2026_elo['team'] == new).sum()
    print(f"  '{old}' remaining: {still_old} | '{new}' rows: {now_new}")

if not WC2026_NAME_FIX:
    print('No fixes applied.')

  'Czechia' remaining: 0 | 'Czech Republic' rows: 1
  'Democratic Republic of Congo' remaining: 0 | 'DR Congo' rows: 1


---
## 6. Save Outputs

In [125]:
results.to_csv(OUTPUT_DIR / 'results_clean.csv', index=False)
hist_elo.to_csv(OUTPUT_DIR / 'historical_elo_clean.csv', index=False)
wc2026_elo.to_csv(OUTPUT_DIR / 'wc2026_elo_clean.csv', index=False)

print('Saved:')
print(f'  output/results_clean.csv         ({len(results):,} rows)')
print(f'  output/historical_elo_clean.csv  ({len(hist_elo):,} rows)')
print(f'  output/wc2026_elo_clean.csv      ({len(wc2026_elo)} teams)')

Saved:
  output/results_clean.csv         (49,378 rows)
  output/historical_elo_clean.csv  (6,646 rows)
  output/wc2026_elo_clean.csv      (48 teams)


---
## What's next?

**Notebook 2 — Form Calculations** will use these outputs to:
- Look up each team's most recent ELO from `historical_elo_clean.csv` as the fixed baseline for form
- Compute overall and competitive form scores for any team over any N matches
- Later: compute form as-of a specific date for historical WC training rows